In [1]:
import pandas as pd
ratings = pd.read_csv(
    "./ml-1m/ratings.dat",
    sep="::",
    engine="python",
    names=["userId", "movieId", "rating", "timestamp"]
)
ratings = ratings[["userId", "movieId", "timestamp", "rating"]]
ratings.head()

,userId,movieId,timestamp,rating
0,1,1193,978300760,5
1,1,661,978302109,3
2,1,914,978301968,3
3,1,3408,978300275,4
4,1,2355,978824291,5


In [2]:
user_ids = ratings["userId"].unique()
movie_ids = ratings["movieId"].unique()

user2idx = {u:i for i, u in enumerate(user_ids)}
movie2idx = {m:i for i, m in enumerate(movie_ids)}

ratings["user"] = ratings["userId"].map(user2idx)
ratings["movie"] = ratings["movieId"].map(movie2idx)

ratings["rating"] = ratings["rating"].astype("float32")
ratings["rating"] = (ratings["rating"] - ratings["rating"].mean()) / ratings["rating"].std()

In [3]:
ratings

,userId,movieId,timestamp,rating,user,movie
0,1,1193,978300760,1.269746,0,0
1,1,661,978302109,-0.520601,0,1
2,1,914,978301968,-0.520601,0,2
3,1,3408,978300275,0.374572,0,3
4,1,2355,978824291,1.269746,0,4
...,...,...,...,...,...,...
1000204,6040,1091,956716541,-2.310948,6039,772
1000205,6040,1094,956704887,1.269746,6039,1106
1000206,6040,562,956704746,1.269746,6039,365
1000207,6040,1096,956715648,0.374572,6039,152


In [4]:
num_users = ratings["user"].nunique()
num_movies = ratings["movie"].nunique()

print(num_users, num_movies)

6040 3706


In [5]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(ratings, test_size = 0.2, random_state = 42)

In [6]:
import torch
import torch.nn as nn
train_user = torch.tensor(train["user"].values)
train_movie = torch.tensor(train["movie"].values)
train_rating = torch.tensor(train["rating"].values, dtype=torch.float32)

In [7]:
class Recommender(nn.Module):

    def __init__(self, num_users, num_movies, emb_size=100):
        super().__init__()

        self.user_emb = nn.Embedding(num_users, emb_size)
        self.movie_emb = nn.Embedding(num_movies, emb_size)

        self.user_bias = nn.Embedding(num_users, 1)
        self.movie_bias = nn.Embedding(num_movies, 1)

        # ADD THIS
        nn.init.normal_(self.user_emb.weight, std=0.01)
        nn.init.normal_(self.movie_emb.weight, std=0.01)
    def forward(self, user, movie):

        u = self.user_emb(user)
        m = self.movie_emb(movie)

        dot = (u * m).sum(1)

        bias = self.user_bias(user).squeeze() + self.movie_bias(movie).squeeze()

        return dot + bias
        #return dot + bias + self.global_bias

In [8]:
model = Recommender(num_users, num_movies)

In [9]:
loss_fn = nn.MSELoss()
initial_lambda = 1e-4
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=initial_lambda
)

for epoch in range(999):

    current_lambda = initial_lambda * (0.95 ** epoch)
    optimizer.param_groups[0]['weight_decay'] = current_lambda

    pred = model(train_user, train_movie)

    loss = loss_fn(pred, train_rating)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    print(f"Epoch {epoch:03d} | Loss {loss.item():.4f}")

Epoch 000 | Loss 3.0062
Epoch 001 | Loss 3.0026
Epoch 002 | Loss 2.9990
Epoch 003 | Loss 2.9954
Epoch 004 | Loss 2.9917
Epoch 005 | Loss 2.9879
Epoch 006 | Loss 2.9841
Epoch 007 | Loss 2.9801
Epoch 008 | Loss 2.9759
Epoch 009 | Loss 2.9715
Epoch 010 | Loss 2.9668
Epoch 011 | Loss 2.9619
Epoch 012 | Loss 2.9567
Epoch 013 | Loss 2.9512
Epoch 014 | Loss 2.9453
Epoch 015 | Loss 2.9391
Epoch 016 | Loss 2.9325
Epoch 017 | Loss 2.9255
Epoch 018 | Loss 2.9181
Epoch 019 | Loss 2.9103
Epoch 020 | Loss 2.9020
Epoch 021 | Loss 2.8932
Epoch 022 | Loss 2.8840
Epoch 023 | Loss 2.8743
Epoch 024 | Loss 2.8640
Epoch 025 | Loss 2.8533
Epoch 026 | Loss 2.8420
Epoch 027 | Loss 2.8302
Epoch 028 | Loss 2.8179
Epoch 029 | Loss 2.8050
Epoch 030 | Loss 2.7916
Epoch 031 | Loss 2.7775
Epoch 032 | Loss 2.7630
Epoch 033 | Loss 2.7478
Epoch 034 | Loss 2.7321
Epoch 035 | Loss 2.7158
Epoch 036 | Loss 2.6990
Epoch 037 | Loss 2.6815
Epoch 038 | Loss 2.6636
Epoch 039 | Loss 2.6450
Epoch 040 | Loss 2.6259
Epoch 041 | Loss

In [10]:
test_user = torch.tensor(test["user"].values)
test_movie = torch.tensor(test["movie"].values)
test_rating = torch.tensor(test["rating"].values, dtype=torch.float32)

In [11]:
import torch.nn.functional as F
import math

model.eval()  # important for evaluation

with torch.no_grad():
    pred = model(test_user, test_movie)
    mse = F.mse_loss(pred, test_rating)
    rmse = math.sqrt(mse.item())

print("Test RMSE:", rmse)

Test RMSE: 0.914126467300504


Evaluation

In [30]:
user_embeddings = model.user_emb.weight.detach().cpu()

print(user_embeddings.shape)

torch.Size([6040, 100])


In [33]:
print(user_embeddings[3])

tensor([ 0.0721, -0.0924, -0.1066,  0.1464, -0.4122, -0.2660,  0.1599,  0.1328,
        -0.1937,  0.2420,  0.3494,  0.0731, -0.1363, -0.1590, -0.2287,  0.2401,
         0.1857, -0.3698, -0.2947,  0.2703,  0.0937, -0.1690,  0.2277, -0.0917,
        -0.5485, -0.0246,  0.1598, -0.1620, -0.1431,  0.1825,  0.3365, -0.0355,
        -0.2490, -0.1247, -0.2984, -0.1976,  0.4073, -0.2837, -0.4723,  0.0527,
        -0.0715,  0.4269, -0.0987,  0.0318, -0.3454, -0.2366, -0.1639,  0.1544,
        -0.1611, -0.2379,  0.0235, -0.2441,  0.0715,  0.1120,  0.1384, -0.2411,
         0.1433,  0.1572,  0.0195, -0.0509,  0.0227, -0.3111, -0.0869, -0.2303,
         0.3437,  0.2533, -0.3188, -0.0228,  0.3722,  0.0842,  0.4860,  0.1882,
         0.3166, -0.0729,  0.2217, -0.0242, -0.2215, -0.2041,  0.2812,  0.1526,
         0.1516,  0.2523,  0.2306,  0.0243,  0.3725,  0.0358, -0.0403, -0.2987,
        -0.0155, -0.4588,  0.1641,  0.1387, -0.1319, -0.1272, -0.1870,  0.1768,
        -0.1986, -0.3649, -0.0916,  0.24

In [35]:
import torch

# Select user 1
user_id = 1

# Get user embedding and bias
user_vec = model.user_emb.weight[user_id]           # shape: [emb_size]
user_b = model.user_bias.weight[user_id].squeeze()  # shape: []

# Get all movie embeddings and biases
movie_vecs = model.movie_emb.weight                 # shape: [num_movies, emb_size]
movie_bs = model.movie_bias.weight.squeeze()       # shape: [num_movies]

In [36]:
# Compute dot product between user and all movies
scores = torch.matmul(movie_vecs, user_vec)  # shape: [num_movies]

# Add biases
scores = scores + user_b + movie_bs 

In [38]:
# Get indices of movies sorted by score descending
ranked_movie_indices = torch.argsort(scores, descending=True)

# Top 10 movies
top10 = ranked_movie_indices[:10]

print("Top 10 recommended movie indices for user 1:", top10.numpy())
print("Corresponding scores:", scores[top10].detach().numpy())

Top 10 recommended movie indices for user 1: [ 597 1950    3 3678 1835 1182 1778  763   92   14]
Corresponding scores: [1.8913913 1.8226599 1.8121932 1.7831657 1.7430341 1.7202293 1.6660523
 1.6557467 1.6163921 1.5836558]


In [41]:
movies = pd.read_csv(
    "ml-1m/movies.dat",
    sep="::",
    engine="python",
    encoding="latin-1",
    names=["movieId", "title", "genres"],
    header=None
)

In [42]:
# Map movieId to index
movie2idx = {m:i for i, m in enumerate(movies["movieId"].unique())}
idx2movie = {i:m for m,i in movie2idx.items()}

# Map index → title
idx2title = {i: movies[movies["movieId"]==mid]["title"].values[0] for i, mid in idx2movie.items()}

In [44]:
# Convert PyTorch tensors to int
top10_int = [int(i) for i in top10]

top10_titles = [idx2title[i] for i in top10_int]

for rank, (title, score) in enumerate(zip(top10_titles, scores[top10].detach().numpy()), start=1):
    print(f"{rank}. {title}  →  score: {score:.3f}")

1. Wooden Man's Bride, The (Wu Kui) (1994)  →  score: 1.891
2. Seven Samurai (The Magnificent Seven) (Shichinin no samurai) (1954)  →  score: 1.823
3. Waiting to Exhale (1995)  →  score: 1.812
4. Jesus' Son (1999)  →  score: 1.783
5. Henry Fool (1997)  →  score: 1.743
6. Aliens (1986)  →  score: 1.720
7. Ratchet (1996)  →  score: 1.666
8. Touki Bouki (Journey of the Hyena) (1973)  →  score: 1.656
9. Beautiful Girls (1996)  →  score: 1.616
10. Cutthroat Island (1995)  →  score: 1.584


In [45]:
# Select test movies for user 1
user_id = 1

liked_movies_indices = test[(test["user"] == user_id) & (test["rating"] > 0)]["movie"].values

In [46]:
# Convert to list of ints
liked_movies_int = [int(i) for i in liked_movies_indices]

# Map to titles
liked_movie_titles = [idx2title[i] for i in liked_movies_int]

print("Movies user 1 liked in test set:")
for title in liked_movie_titles:
    print("-", title)

Movies user 1 liked in test set:
- Unforgettable (1996)
- Bad Boys (1995)
- Neon Bible, The (1995)
- NeverEnding Story III, The (1994)
- Die Hard: With a Vengeance (1995)
- Congo (1995)
- Big Green, The (1995)
- Two if by Sea (1996)
- Georgia (1995)


In [ ]:
recalls = []
for user_id in test["user"].unique():
    liked_movies = test[(test["user"]==user_id) & (test["rating"]>0)]["movie"].values
    if len(liked_movies)==0:
        continue

    all_movies = torch.arange(num_movies)
    user_tensor = torch.tensor([user_id]*num_movies)
    with torch.no_grad():
        scores = model(user_tensor, all_movies)

    seen_movies = train[train["user"]==user_id]["movie"].values
    scores[seen_movies] = -1e9

    top10 = torch.topk(scores, 10).indices.numpy()
    hits = len(set(top10) & set(liked_movies))
    recalls.append(hits / len(liked_movies))

average_recall = sum(recalls)/len(recalls)
print("Average Recall@10:", average_recall)

/tmp/ipykernel_46120/767896117.py:13: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  scores[seen_movies] = -1e9


Average Recall@10: 0.007888222272899603


In [47]:
import torch

K = 10
recalls = []

# For each user
for user_id in ratings["user"].unique():

    # Get all movies rated by this user
    user_movies = ratings[ratings["user"]==user_id].sort_values("timestamp")

    # Leave the last one for test
    test_movie = user_movies.iloc[-1]["movie"]
    train_movies = user_movies.iloc[:-1]["movie"].values

    # Prepare all movies for scoring
    all_movies = torch.arange(num_movies)
    user_tensor = torch.tensor([user_id]*num_movies)

    with torch.no_grad():
        scores = model(user_tensor, all_movies)

    # Remove movies seen in train
    scores[train_movies] = -1e9

    # Top-K recommendations
    topk = torch.topk(scores, K).indices.numpy()

    # Check if the left-out test movie is in top-K
    hit = 1 if test_movie in topk else 0
    recalls.append(hit)

# Average Recall@K
avg_recall_at_10 = sum(recalls)/len(recalls)
print("Average Recall@10 (Leave-One-Out):", avg_recall_at_10)

Average Recall@10 (Leave-One-Out): 0.002980132450331126
